<a href="https://colab.research.google.com/github/Lukas98275/Mineria-de-datos/blob/main/fc26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Origen y naturaleza:**
El dataset proviene de Kaggle (rovnez/fc-26-fifa-26-player-data) y recopila información estadística, demográfica y contractual de los futbolistas profesionales incluidos en la base de datos oficial del videojuego. Cada fila original del archivo CSV representa a un jugador individual, detallando atributos como su nombre, posición, valoración general, club de adscripción, liga, nacionalidad, entre otros.

# **Definición de Transacción:**
Transacción (Fila): Una nacionalidad específica de jugadores (por ejemplo, Argentina, Francia, España).

Ítem (Columna): Un club de fútbol (por ejemplo, Chelsea, Fulham FC, AS Monaco).

Valor: Un valor booleano (True si hay al menos un jugador de esa nacionalidad específica registrado en ese club, y False en caso contrario).

# **Volumen y dimensiones:**

# **Transacciones totales:**
Equivalen al número total de clubes únicos analizados (cientos de equipos oficiales de distintas ligas mundiales).

# **Ítems distintos:**
Equivalen al universo total de futbolistas únicos considerados en el estudio.

# **Estructura final (Matriz One-Hot):**
 Tras limpiar valores nulos y descartar registros de agentes libres (Free Agents), los datos se transforman en una matriz binaria dispersa donde las filas son los clubes y las columnas son los jugadores, conteniendo valores booleanos (True/False) que indican la presencia o ausencia de un futbolista determinado en la plantilla de ese club.

In [1]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

from google.colab import files
archivos_subidos = files.upload()
nombre_csv = next(iter(archivos_subidos))
print("Archivo seleccionado:", nombre_csv)

Saving FC26_20250921.csv to FC26_20250921 (1).csv
Archivo seleccionado: FC26_20250921 (1).csv


In [2]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset usando el nombre del archivo que ya subiste
df = pd.read_csv(nombre_csv)

print("Dimensiones originales del dataset:", df.shape)
print("Columnas disponibles:", df.columns.tolist())

# 2. Limpieza básica: Filtrar valores nulos en nacionalidad y club
# Nota: Los nombres exactos de las columnas pueden variar, comúnmente son 'nationality_name' y 'club_name' o similares.
# Verificaremos las columnas primero si es necesario, pero asumiremos nombres estándar de datasets de FIFA.
col_nacion = 'nationality_name' if 'nationality_name' in df.columns else [c for c in df.columns if 'nation' in c.lower()][0]
col_club = 'club_name' if 'club_name' in df.columns else [c for c in df.columns if 'club' in c.lower()][0]

df_clean = df[[col_nacion, col_club]].dropna()

# Eliminar nombres inconsistentes o vacíos
df_clean = df_clean[(df_clean[col_nacion] != '') & (df_clean[col_club] != '')]

# 3. Estructura Transaccional: Nacionalidad (Fila) x Club (Columna)
# Creamos una tabla cruzada donde el valor indica si existe al menos un jugador de esa nacionalidad en el club
transaccional = pd.crosstab(df_clean[col_nacion], df_clean[col_club])

# Convertir a valores booleanos (True/False)
transaccional_bool = transaccional > 0

print(f"Número de transacciones (Nacionalidades): {transaccional_bool.shape[0]}")
print(f"Número de ítems (Clubes): {transaccional_bool.shape[1]}")

# Mostrar las primeras filas de la matriz transaccional
transaccional_bool.head()

Dimensiones originales del dataset: (18405, 110)
Columnas disponibles: ['player_id', 'player_url', 'fifa_version', 'fifa_update', 'fifa_update_date', 'short_name', 'long_name', 'player_positions', 'overall', 'potential', 'value_eur', 'wage_eur', 'age', 'dob', 'height_cm', 'weight_kg', 'league_id', 'league_name', 'league_level', 'club_team_id', 'club_name', 'club_position', 'club_jersey_number', 'club_loaned_from', 'club_joined_date', 'club_contract_valid_until_year', 'nationality_id', 'nationality_name', 'nation_team_id', 'nation_position', 'nation_jersey_number', 'preferred_foot', 'weak_foot', 'skill_moves', 'international_reputation', 'work_rate', 'body_type', 'real_face', 'release_clause_eur', 'player_tags', 'player_traits', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy', 'attacking_short_passing', 'attacking_volleys', 'skill_dribbling', 'skill_curve', 'skill_fk_accuracy', 'skill_long_passi

/tmp/ipykernel_9295/1918895677.py:5: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nombre_csv)


club_name,1. FC Heidenheim 1846,1. FC Kaiserslautern,1. FC Köln,1. FC Magdeburg,1. FC Nürnberg,1. FC Saarbrücken,1. FC Schweinfurt 05,1. FC Union Berlin,1. FSV Mainz 05,AC Milan,...,Wolfsberger AC,Wolverhampton Wanderers,Wrexham,Wuhan Three Towns,Wycombe Wanderers,Yunnan Yukun,Zagłębie Lubin,Zhejiang FC,Çaykur Rizespor,Östers IF
nationality_name,,,,,,,,,,,,,,,,,,,,,
Afghanistan,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
Albania,False,True,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
Algeria,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
Andorra,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
Angola,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False


### **¿Qué es una transacción y un ítem?** Una transacción es una nacionalidad específica de jugadores (por ejemplo, Argentina) y un ítem es un club de fútbol.

### **¿Por qué Apriori necesita este formato?** El algoritmo Apriori requiere una matriz dispersa (sparse matrix) de tipo booleana o binaria (True/False o 1/0), donde cada fila representa una cesta/transacción y cada columna un producto/ítem posible, permitiendo calcular eficientemente qué ítems cocurren frecuentemente en las transacciones.

In [7]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
# Instalar mlxtend si no está disponible
try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError:
    !pip install mlxtend
    from mlxtend.frequent_patterns import apriori, association_rules

# Experimento obligatorio con al menos 3 combinaciones de soporte y confianza
configuraciones = [
    {"support": 0.05, "confidence": 0.6},
    {"support": 0.10, "confidence": 0.7},
    {"support": 0.15, "confidence": 0.8}
]

resultados_comparativos = []

for i, cfg in enumerate(configuraciones, 1):
    # Obtener itemsets frecuentes
    frequent_itemsets = apriori(transaccional_bool, min_support=cfg["support"], use_colnames=True)

    # Validar si el DataFrame de itemsets frecuentes está vacío antes de generar reglas
    if not frequent_itemsets.empty:
        rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=cfg["confidence"])
        num_rules = len(rules)
    else:
        num_rules = 0

    resultados_comparativos.append({
        "Configuración": f"#{i} (Sup: {cfg['support']}, Conf: {cfg['confidence']})",
        "Itemsets Frecuentes": len(frequent_itemsets),
        "Reglas Generadas": num_rules
    })

# Crear la tabla comparativa solicitada
df_comparacion = pd.DataFrame(resultados_comparativos)
print("TABLA COMPARATIVA DE CONFIGURACIONES:")
display(df_comparacion)

# Generar las reglas detalladas con la configuración óptima (Configuración 1 ya que tiene resultados)
cfg_optima = configuraciones[0]
frequent_itemsets_opt = apriori(transaccional_bool, min_support=cfg_optima["support"], use_colnames=True)

if not frequent_itemsets_opt.empty:
    rules_opt = association_rules(frequent_itemsets_opt, metric="confidence", min_threshold=cfg_optima["confidence"])
    if not rules_opt.empty:
        # Ordenar reglas por Lift de manera descendente
        rules_opt = rules_opt.sort_values(by="lift", ascending=False)
        print("\nPrimeras reglas generadas (ordenadas por Lift):")
        display(rules_opt[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))
    else:
        print("\nNo se generaron reglas de asociación para la configuración óptima seleccionada.")
else:
    print("\nNo se encontraron itemsets frecuentes para la configuración óptima seleccionada.")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

TABLA COMPARATIVA DE CONFIGURACIONES:


,Configuración,Itemsets Frecuentes,Reglas Generadas
0,"#1 (Sup: 0.05, Conf: 0.6)",981,353
1,"#2 (Sup: 0.1, Conf: 0.7)",59,0
2,"#3 (Sup: 0.15, Conf: 0.8)",0,0



Primeras reglas generadas (ordenadas por Lift):


,antecedents,consequents,support,confidence,lift
349,"(Chelsea, AS Monaco)","(Fulham FC, Olympique de Marseille)",0.050314,1.000000,19.875000
348,"(Fulham FC, Olympique de Marseille)","(Chelsea, AS Monaco)",0.050314,1.000000,19.875000
347,"(Olympique de Marseille, AS Monaco)","(Fulham FC, Chelsea)",0.050314,0.888889,17.666667
350,"(Fulham FC, Chelsea)","(Olympique de Marseille, AS Monaco)",0.050314,1.000000,17.666667
352,(Chelsea),"(Fulham FC, Olympique de Marseille, AS Monaco)",0.050314,0.727273,14.454545
306,(Chelsea),"(Fulham FC, Olympique de Marseille)",0.050314,0.727273,14.454545
304,"(Fulham FC, Olympique de Marseille)",(Chelsea),0.050314,1.000000,14.454545
344,"(Fulham FC, Olympique de Marseille, AS Monaco)",(Chelsea),0.050314,1.000000,14.454545
351,"(Fulham FC, AS Monaco)","(Chelsea, Olympique de Marseille)",0.050314,0.800000,14.133333
346,"(Chelsea, Olympique de Marseille)","(Fulham FC, AS Monaco)",0.050314,0.888889,14.133333


# **Interpretación de las 5 reglas principales:**

1.-
Red compartida anglo-francesa: Si una nacionalidad abastece simultáneamente al Chelsea y al AS Monaco, existe una certeza total (confianza del 100%) de que sus futbolistas también militan en el Fulham FC y el Olympique de Marseille.

2.-
Ecosistema cruzado de ligas: Las nacionalidades que logran posicionar jugadores de manera conjunta en el Fulham FC y el Olympique de Marseille tienen garantizada su presencia simultánea en los planteles del Chelsea y el AS Monaco.

3.-
Sinergia mediterránea-atlántica: La presencia de una nacionalidad en el Olympique de Marseille y el AS Monaco indica una probabilidad del 88.8% de que también exporte talento al Fulham FC y al Chelsea.

4.-
Alianza de clubes principales: Cuando los futbolistas de una nacionalidad cubren el Fulham FC y el Chelsea, se cumple infaliblemente (confianza 1.0) que también cubren al Olympique de Marseille y al AS Monaco.

5.-
Concentración en Londres y Marsella: Con un alto lift de 14.45, la presencia de una nacionalidad en el Chelsea predice con un 72.7% de confianza su presencia conjunta en el Fulham FC y el Olympique de Marseille.

# **Regla descartada y su justificación:**

A pesar de que las reglas generadas poseen métricas de confianza elevadas (algunas del 100%) y un lift superior a 14, se descartarían aquellas reglas donde el antecedente provenga exclusivamente de un sesgo de nacionalidades hiper-masivas (como jugadores de Brasil o Argentina en cualquier club top europeo). Aunque matemáticamente superen los umbrales, su alto soporte absoluto y confianza se deben a la popularidad general de dichos países en el simulador y no a una estrategia de scouting oculta, por lo que no aportan un valor diferenciado para la toma de decisiones

# **Decisión concreta para un administrador:**
Un director deportivo o administrador de scouting puede utilizar estas reglas para establecer un modelo predictivo de fichajes: si el club detecta talento emergente de una nacionalidad específica que empieza a destacar en clubes puente (como el AS Monaco o el Olympique de Marseille), la directiva debe anticiparse y ojear activamente a esa misma nacionalidad para los planteles de la Premier League (como Chelsea o Fulham), aprovechando los corredores migratorios del fútbol internacional identificados por el algoritmo.